# BOSCH — DroneSplat + Generative 3D Distillation (LLMs) Inference Pipeline

> **Kernel**: `dronesplat_llms` (Set via Kernel → Change Kernel after running `bosch_setup_dronesplat_llms.ipynb`)

Runs the integrated DroneSplat + NVIDIA Difix3D+ 3D Geometry Distillation pipeline on competition scenes (ArtiFixer is not yet wired in — see Cell 1).
Supervises 3D Gaussian Splatting optimization with generative 3D diffusion priors to eliminate blur, floaters, and sparse-view artifacts on the main BTS object.

| Cell | Purpose |
|------|---------|
| 0 | **Proxy** — BOSCH proxy env vars for this kernel process |
| 1 | **CONFIG** — Paths, scenes, hyper-parameters, and distillation toggles |
| 2 | **GPU & Imports Check** — Hardware detection (A100-80GB) and module validation (incl. diffusers/pipeline_difix) |
| 3 | **Dataset Verification & Workspace** — Reuse DATA_ROOT directly when writable, symlink workspace as fallback |
| 4 | **Train List Generation** — Exclude competition test images and write splits |
| 4.5 | **SAM2 Mask Generation** — Download SAM2 checkpoint, run `seg_all_instances.py` per scene for `masks.json` |
| 5 | **DroneSplat + Generative 3D Distillation Training** — Trains Gaussians with live diffusion feedback & visual logging |
| 6 | **Novel View Rendering + Difix3D+ Cleanup** — Render competition test viewpoints with `render_competition.py --enable_difix` |
| 7 | **Package Submission ZIP** — Build `submission_round1_dronesplat_llms.zip` |
| 8 | **Save Logs & Artifacts ZIP** — Save intermediate visual grids, metric histories, and PLYs |

## Cell 0 — Proxy
Sets the BOSCH proxy for this kernel process. `bosch_setup_dronesplat_llms.ipynb` sets it too, but that's a separate kernel process — these env vars don't carry over, and Difix3D+ (`nvidia/difix_ref`, loaded with `trust_remote_code=True`) can still need network access even with cached weights. Run this first.

In [ ]:
# ── Proxy (required for HuggingFace / SAM2 checkpoint downloads on BOSCH) ────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')
HF_HOME = os.path.join(HOME, '.cache/huggingface')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY
os.environ['HF_HOME']     = HF_HOME

print(f'Proxy set: {PROXY}')
print(f'HF_HOME  : {HF_HOME}')

## Cell 1 — CONFIG

In [ ]:
import os, sys

# ── Paths ─────────────────────────────────────────────────────────────────────
HOME       = os.path.expanduser('~')
REPO_DIR   = f'{HOME}/Race-AI-2026'
DRONESPLAT = f'{REPO_DIR}/projects/DroneSplat'
DIFIX_DIR  = f'{REPO_DIR}/projects/Difix3D'
SCRIPTS    = f'{REPO_DIR}/scripts'

DATA_ROOT  = f'{HOME}/data/phase1/private_set1'
if not os.path.isdir(DATA_ROOT):
    DATA_ROOT = f'{HOME}/data/private_set1'

# Output directories for trained model, logs, and rendered submission
RUN_DIR        = f'{HOME}/dronesplat_llms_run'
LOCAL_DATA_DIR = f'{RUN_DIR}/data'
OUTPUT_ROOT    = f'{RUN_DIR}/output'
RENDERS_ROOT   = f'{RUN_DIR}/renders'
SUBMISSION_DIR = f'{RUN_DIR}/submission'
LOGS_DIR       = f'{RUN_DIR}/distillation_logs'

# ── Dataset Scenes ────────────────────────────────────────────────────────────
SPLIT  = 'private_set1'
SCENES = [
    'HCM0249', 'HCM0254', 'HCM0276', 'HCM1439',
    'HNI0131', 'HNI0265', 'HNI0366', 'HNI0437'
]

# ── Training & Generative 3D Distillation Parameters ─────────────────────────
ITERATIONS              = 7000
USE_MASKS               = True    # wired into Cell 4.5 (SAM2 mask gen) and Cell 5 (--use_masks)
ENABLE_DIFIX3D_DISTILL   = True   # wired into Cell 5 (--enable_distill / --no-enable_distill)
# ArtiFixer is NOT wired into any script this notebook calls -- artifixer_distill_trainer.py
# is never invoked here, so there is currently nothing for this flag to enable. Left as a
# reminder of intended scope, not a working toggle. See scripts/artifixer_distill_trainer.py.
ENABLE_ARTIFIXER_DISTILL_NOT_YET_WIRED = True
LOG_VISUAL_FREQ         = 500    # Frequency to record 2x2 visual comparison grids

print(f'Active Python     : {sys.executable}')
print(f'DATA_ROOT         : {DATA_ROOT}')
print(f'RUN_DIR           : {RUN_DIR}')
print(f'LOGS_DIR          : {LOGS_DIR}')
print(f'DRONESPLAT        : {DRONESPLAT}')
print(f'ITERATIONS        : {ITERATIONS}')
print(f'Difix3D+ Distill  : {ENABLE_DIFIX3D_DISTILL}')
print(f'ArtiFixer Distill : not wired into this pipeline yet')
print(f'Scenes count      : {len(SCENES)} scenes')

## Cell 2 — GPU check + imports

In [ ]:
import sys, subprocess
sys.path.insert(0, DRONESPLAT)
sys.path.insert(0, SCRIPTS)
sys.path.insert(0, os.path.join(DIFIX_DIR, 'src'))

import torch
print(f'torch         : {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')

GPU_COUNT = torch.cuda.device_count()
print(f'GPU count     : {GPU_COUNT}')
for i in range(GPU_COUNT):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name}  ({p.total_memory / 1e9:.1f} GB)')

import diff_gaussian_rasterization
import simple_knn
import distillation_logger
import diffusers
from pipeline_difix import DifixPipeline
print('diff_gaussian_rasterization: OK')
print('simple_knn                 : OK')
print('distillation_logger        : OK')
print(f'diffusers                  : OK ({diffusers.__version__})')
print('pipeline_difix              : OK')

## Cell 3 — Collect scenes, verify dataset & create writeable local workspace

In [ ]:
import os
found_scenes = []

# Reuse DATA_ROOT directly when it's writable (train_list.txt/test_list.txt and
# generated masks can be written in place) -- avoids symlinking a second copy of
# the dataset into RUN_DIR/data when there's no need to.
DATA_ROOT_WRITABLE = os.access(DATA_ROOT, os.W_OK)
SCENE_TRAIN_DIR = {}

if not DATA_ROOT_WRITABLE:
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)

for s in SCENES:
    s_path = os.path.join(DATA_ROOT, s)
    if not os.path.isdir(s_path):
        print(f'  WARNING: Scene directory missing -> {s_path}')
        continue
    found_scenes.append(s)

    if DATA_ROOT_WRITABLE:
        SCENE_TRAIN_DIR[s] = os.path.join(s_path, 'train')
        print(f'  [OK] Using dataset in place for: {s}')
        continue

    # DATA_ROOT is read-only -- fall back to a writeable symlinked workspace.
    local_s = os.path.join(LOCAL_DATA_DIR, s)
    os.makedirs(local_s, exist_ok=True)

    src_test = os.path.join(s_path, 'test')
    dst_test = os.path.join(local_s, 'test')
    if os.path.exists(src_test) and not os.path.exists(dst_test):
        os.symlink(src_test, dst_test)

    src_train = os.path.join(s_path, 'train')
    dst_train = os.path.join(local_s, 'train')
    os.makedirs(dst_train, exist_ok=True)
    if os.path.exists(src_train):
        for child in os.listdir(src_train):
            sc = os.path.join(src_train, child)
            dc = os.path.join(dst_train, child)
            if not os.path.exists(dc):
                os.symlink(sc, dc)
    SCENE_TRAIN_DIR[s] = dst_train
    print(f'  [OK] Found and configured workspace for: {s}')

workspace_note = 'training writes directly into DATA_ROOT' if DATA_ROOT_WRITABLE else f'using symlinked workspace at {LOCAL_DATA_DIR}'
print(f'Total valid scenes: {len(found_scenes)} / {len(SCENES)}')
print(f'DATA_ROOT writable : {DATA_ROOT_WRITABLE} ({workspace_note})')

## Cell 4 — Generate train_list.txt for each scene

In [ ]:
import csv, os

for scene in found_scenes:
    train_root       = SCENE_TRAIN_DIR[scene]
    train_dir        = os.path.join(train_root, 'images')
    test_csv         = os.path.join(DATA_ROOT, scene, 'test', 'test_poses.csv')
    train_list_path  = os.path.join(train_root, 'train_list.txt')
    test_list_path   = os.path.join(train_root, 'test_list.txt')
    
    if os.path.isdir(train_dir) and os.path.isfile(test_csv):
        with open(test_csv) as f:
            comp_names = {r['image_name'].strip() for r in csv.DictReader(f)}
        
        all_imgs = sorted([fn for fn in os.listdir(train_dir) if fn.lower().endswith(('.jpg','.jpeg','.png')) and fn not in comp_names])
        stems = [os.path.splitext(n)[0] for n in all_imgs]
        
        with open(train_list_path, 'w') as f:
            f.write('\n'.join(stems))
            
        n_holdout = max(1, len(stems) // 10)
        with open(test_list_path, 'w') as f:
            f.write('\n'.join(stems[-n_holdout:]))
            
        print(f'{scene}: {len(stems)} train stems -> {train_list_path}')

## Cell 4.5 — SAM2 checkpoint download & instance mask generation
Downloads the SAM2 checkpoint and runs `seg_all_instances.py` per scene to produce `masks/masks.json`, so `difix3d_distill_trainer.py`'s `--use_masks` (distractor/moving-object down-weighting) actually has masks to read. Ported from `bosch_inference_dronesplat.ipynb` Cell 3.5, which this notebook was previously missing. Skips scenes that already have `masks.json`.

In [ ]:
# ── SAM2 checkpoint download + instance segmentation mask generation ─────────
import os, subprocess, sys, urllib.request, time

if USE_MASKS:
    checkpoint_dir = os.path.join(DRONESPLAT, 'checkpoints')
    os.makedirs(checkpoint_dir, exist_ok=True)
    checkpoint_path = os.path.join(checkpoint_dir, 'sam2_hiera_large.pt')

    if not os.path.exists(checkpoint_path):
        url = "https://dl.fbaipublicfiles.com/segment_anything_2/072824/sam2_hiera_large.pt"
        print(f'Downloading SAM2 checkpoint from {url} ...')
        proxy_handler = urllib.request.ProxyHandler({'http': PROXY, 'https': PROXY})
        opener = urllib.request.build_opener(proxy_handler)
        urllib.request.install_opener(opener)

        def _reporthook(blocknum, blocksize, totalsize):
            read = blocknum * blocksize
            if totalsize > 0:
                pct = read * 1e2 / totalsize
                sys.stdout.write(f'\r  {pct:.1f}% ({read/1e6:.1f}MB / {totalsize/1e6:.1f}MB)')
                sys.stdout.flush()

        urllib.request.urlretrieve(url, checkpoint_path, _reporthook)
        print('\nDownload complete.')
    else:
        print(f'SAM2 checkpoint already present at {checkpoint_path}')

    print('\nGenerating masks.json for each scene (SAM2 instance segmentation)...')
    for scene in found_scenes:
        scene_train = SCENE_TRAIN_DIR[scene]
        masks_json = os.path.join(scene_train, 'masks', 'masks.json')

        if os.path.exists(masks_json):
            print(f'  [OK] Masks already exist for {scene}')
            continue

        print(f'  Segmenting {scene} ...')
        cmd = [
            sys.executable, 'seg_all_instances.py',
            '--image_dir', scene_train,
            '--model_checkpoint', checkpoint_path,
            '--model_cfg', 'sam2_hiera_l.yaml',
        ]
        t0 = time.time()
        r = subprocess.run(cmd, cwd=DRONESPLAT)
        elapsed = time.time() - t0
        if r.returncode != 0:
            print(f'    [ERROR] Segmentation failed for {scene} (rc={r.returncode})')
        else:
            print(f'    Done in {elapsed:.1f}s')
else:
    print('USE_MASKS is False -- skipping SAM2 mask generation.')

## Cell 5 — Train DroneSplat + Generative 3D Distillation (Difix3D+)

In [ ]:
# ── Train with Full 3D Geometry Distillation & Visual Logging ─────────────────
import subprocess, sys, time

os.makedirs(OUTPUT_ROOT, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

py_bin = sys.executable

for idx, scene in enumerate(found_scenes):
    print(f'\n[Scene {idx+1}/{len(found_scenes)}] Launching DroneSplat + 3D Geometry Distillation for {scene}...')
    s_data = SCENE_TRAIN_DIR[scene]
    s_out  = os.path.join(OUTPUT_ROOT, scene)
    
    train_cmd = [
        py_bin, f'{SCRIPTS}/difix3d_distill_trainer.py',
        '--source_path', s_data,
        '--model_path', s_out,
        '--log_dir', LOGS_DIR,
        '--iterations', str(ITERATIONS),
        '--save_freq', str(LOG_VISUAL_FREQ),
        '--exp_name', f'{scene}_distill_run',
        '--enable_distill' if ENABLE_DIFIX3D_DISTILL else '--no-enable_distill',
        '--use_masks' if USE_MASKS else '--no-use_masks',
    ]
    
    t0 = time.time()
    r = subprocess.run(train_cmd, capture_output=True, text=True)
    elapsed = time.time() - t0
    print(f'Finished {scene} in {elapsed/60:.1f} minutes.')
    if r.returncode != 0:
        print(f'STDERR for {scene}:', r.stderr[-1000:])
    else:
        print(f'Training & 3D Distillation OK for {scene}')

## Cell 6 — Render competition novel views + reference-conditioned Difix3D+ cleanup
`--enable_difix` is on: each rendered pose is cleaned with `nvidia/difix_ref`, conditioned on the training view nearest by position+direction, then fused with `raw + 0.16*(difix-raw)`. Debug artifacts (chosen reference, Difix output, selection metadata) are written to `<scene>/_difix_debug/` inside the submission dir.

In [ ]:
py_bin = sys.executable
os.makedirs(RENDERS_ROOT, exist_ok=True)
os.makedirs(SUBMISSION_DIR, exist_ok=True)

for scene in found_scenes:
    s_out   = os.path.join(OUTPUT_ROOT, scene)
    s_csv   = os.path.join(DATA_ROOT, scene, 'test', 'test_poses.csv')
    s_rend  = os.path.join(SUBMISSION_DIR, scene)
    s_train = SCENE_TRAIN_DIR[scene]
    os.makedirs(s_rend, exist_ok=True)
    
    print(f'Rendering competition novel views for {scene}...')
    render_cmd = [
        py_bin, f'{DRONESPLAT}/render_competition.py',
        '--model_path', s_out,
        '--test_poses_csv', s_csv,
        '--output_dir', s_rend,
        '--iteration', str(ITERATIONS),
        '--enable_difix',
        '--train_source_path', s_train,
    ]
    r = subprocess.run(render_cmd, capture_output=True, text=True)
    if r.returncode != 0:
        print(f'  [ERROR] Rendering FAILED for {scene}:\n', r.stderr[-1000:])
    else:
        print(f'  [OK] Render completed for {scene} -> {s_rend}')

## Cell 7 — Package submission ZIP

In [ ]:
import shutil
zip_out = os.path.join(RUN_DIR, 'submission_round1_dronesplat_llms')
shutil.make_archive(zip_out, 'zip', SUBMISSION_DIR)
print(f'Created competition submission archive: {zip_out}.zip')

## Cell 8 — Package logs & visual artifacts ZIP

In [ ]:
artifacts_zip = os.path.join(RUN_DIR, 'distillation_logs_artifacts')
if os.path.isdir(LOGS_DIR):
    shutil.make_archive(artifacts_zip, 'zip', LOGS_DIR)
    print(f'Created log & visual artifacts archive: {artifacts_zip}.zip')
else:
    print('No distillation logs directory found to archive.')